# 02_transformar_ventas
Limpieza de ventas_bronze y generacion de ventas_silver y ventas_por_categoria.

In [0]:
from pyspark.sql.functions import col, trim, sum as _sum, count, round as _round

ESQ = "workspace.ingenieria_datos_pedro"
bronze = spark.table(f"{ESQ}.ventas_bronze")
registros_antes = bronze.count()
print("Registros antes de la limpieza:", registros_antes)

Registros antes de la limpieza: 16


In [0]:
sin_duplicados = bronze.dropDuplicates()
print("Registros despues de quitar duplicados:", sin_duplicados.count())

ventas_validas = sin_duplicados.filter(
    col("producto").isNotNull()
    & (trim(col("producto")) != "")
    & (col("cantidad") > 0)
    & (col("precio_unitario") > 0)
)

ventas_silver = ventas_validas.withColumn("total_venta", col("cantidad") * col("precio_unitario"))
print("Registros despues de la limpieza:", ventas_silver.count())
display(ventas_silver.orderBy("id_venta"))

Registros despues de quitar duplicados: 15
Registros despues de la limpieza: 13


id_venta,fecha,producto,categoria,cantidad,precio_unitario,total_venta
1,2026-10-01,Audifonos,Electronica,2,85000.0,170000.0
2,2026-10-01,Camiseta,Ropa,3,45000.0,135000.0
3,2026-10-02,Licuadora,Hogar,1,220000.0,220000.0
4,2026-10-02,Arroz 5kg,Alimentos,4,18000.0,72000.0
5,2026-10-03,Mouse inalambrico,Electronica,2,60000.0,120000.0
6,2026-10-03,Jean,Ropa,1,120000.0,120000.0
7,2026-10-04,Sarten,Hogar,2,75000.0,150000.0
8,2026-10-04,Cafe 500g,Alimentos,6,22000.0,132000.0
9,2026-10-05,Teclado,Electronica,1,150000.0,150000.0
10,2026-10-05,Chaqueta,Ropa,1,250000.0,250000.0


In [0]:
# overwrite: cada ejecucion reemplaza el contenido, asi no se acumulan duplicados
ventas_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{ESQ}.ventas_silver")

ventas_por_categoria = (ventas_silver.groupBy("categoria")
    .agg(count("*").alias("numero_ventas"), _round(_sum("total_venta"), 2).alias("total_vendido"))
    .orderBy(col("total_vendido").desc()))
ventas_por_categoria.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{ESQ}.ventas_por_categoria")
display(spark.table(f"{ESQ}.ventas_por_categoria"))

categoria,numero_ventas,total_vendido
Electronica,4,560000.0
Ropa,3,505000.0
Hogar,3,500000.0
Alimentos,3,249000.0


In [0]:
print("ventas_bronze:", spark.table(f"{ESQ}.ventas_bronze").count())
print("ventas_silver:", spark.table(f"{ESQ}.ventas_silver").count())
print("ventas_por_categoria:", spark.table(f"{ESQ}.ventas_por_categoria").count())

ventas_bronze: 16
ventas_silver: 13
ventas_por_categoria: 4
